# Render the kiisafezone header SVG 🎬

This notebook clones the latest public GitHub `main` branch and renders **only `assets/header.svg`** as a standalone MP4. The original SVG animation is kept intact and its timeline is played at 0.5× speed. The 28-second output covers the SVG's 14-second animation timeline.

**Output:** 1080 × 324, 30 FPS, 28 seconds. Run all cells from top to bottom.

In [ ]:
# Install Node.js 22 and FFmpeg in this temporary Colab runtime
!apt-get update -qq
!apt-get install -y -qq curl ca-certificates ffmpeg
!curl -fsSL https://deb.nodesource.com/setup_22.x -o /tmp/nodesource_setup.sh
!bash /tmp/nodesource_setup.sh
!apt-get install -y -qq nodejs
!node --version && npm --version && ffmpeg -version | head -n 1

In [ ]:
# Download the latest main branch and install the pinned Remotion dependencies
%cd /content
!rm -rf /content/kiisafezone
!git clone --depth 1 https://github.com/kiisafezone/kiisafezone.git
%cd /content/kiisafezone/showcase
!npm install --no-audit --no-fund

## Alternative renderer test: `svg-to-video` (Rust)

This section tests the original animated SVG independently of Remotion. It renders the native 14-second SVG timeline at 1080 × 324 and 30 FPS, then slows the result to 0.5× (28 seconds).

In [ ]:
# Install Chromium and the Rust-based svg-to-video CLI
!apt-get update -qq
!apt-get install -y -qq chromium ffmpeg build-essential pkg-config libssl-dev
!curl https://sh.rustup.rs -sSf | sh -s -- -y
%env PATH=/root/.cargo/bin:$PATH
!cargo install svg-to-video --locked
!chromium --version
!svg-to-video --help

In [ ]:
# Render the original SVG at native speed first (14 seconds)
%cd /content/kiisafezone/showcase
!mkdir -p out
!rm -f out/header-native.mp4 out/header-svg-to-video.mp4
!svg-to-video video public/assets/header.svg out/header-native.mp4 --duration 14 --fps 30 --width 1080 --height 324 --codec h264

# Slow the rendered video to 0.5× without changing its frame rate
!ffmpeg -y -i out/header-native.mp4 -vf "setpts=2*PTS" -an -c:v libx264 -crf 18 -preset medium -pix_fmt yuv420p out/header-svg-to-video.mp4
!ls -lh out/header-native.mp4 out/header-svg-to-video.mp4

In [ ]:
# Download the svg-to-video test result
from google.colab import files
files.download('/content/kiisafezone/showcase/out/header-svg-to-video.mp4')

In [ ]:
# Render only the original header.svg (no browser frame or README elements)
!mkdir -p out
!rm -f out/header-only.mp4
!npx remotion render src/index.ts HeaderSvgOnly out/header-only.mp4 --codec=h264 --crf=18 --concurrency=2
!ls -lh out/header-only.mp4

In [ ]:
# Download the standalone header MP4
from google.colab import files
files.download('/content/kiisafezone/showcase/out/header-only.mp4')